name : mohammadmahdi yari

sid : 810102532

In [2]:
from game import Game
import copy
from collections import deque
import heapq

map example:

In [1]:
game_map = """W   W   W   W   W   W   W   W   W   W   W   W
W   .   .   .   .   .   .   W   .   .   .   W
W   .   .   .   B2  .   .   W   .   G1  .   W
W   .   .   .   .   .   .   W   G4  P1  G2  W
W   .   .   .   .   .   .   W   .   G3  .   W
W   W   W   W   W   W   .   W   W   W   W   W
W   .   .   .   .   .   .   .   .   .   .   W
W   .   .   .   .   B4  .   .   .   .   .   W
W   .   .   H   B3  .   .   P1  B1  .   .   W
W   .   .   .   .   .   .   .   .   .   .   W
W   .   .   .   .   .   .   .   .   .   .   W
W   W   W   W   W   W   W   W   W   W   W   W"""
dir_map = {'U': (-1, 0), 'D': (1, 0), 'L': (0, -1), 'R': (0, 1)}


In [423]:
def bfs(game_map):
    game = Game(game_map)
    queue = deque([(game.get_player_position(), tuple(game.get_box_locations()), "")])
    mark = set()
    while queue:
        mike, boxes, path = queue.popleft()
        if boxes == tuple(game.get_goal_locations()):
            return path, len(mark)
        s = (mike, boxes)
        if s in mark:
            continue
        mark.add(s)
        for d in dir_map:
            game.set_player_position(mike)
            game.set_box_positions(list(boxes))
            if game.apply_move(d):
                neeew = (game.get_player_position(), tuple(game.get_box_locations()), path + d)
                queue.append(neeew)
    return "No solution", len(mark)




We implemented this algorithm using a queue, where for each node we put its children in the queue and expand them in a superficial order.

In [435]:
path, state = bfs(game_map)
print(path)
print(state) 

RURRDDDDLDRUUUULLLRDRDRDDLLDLLUUDR
644753


In [228]:
def dfs(game_map):
    game = Game(game_map)
    stack = [(game.get_player_position(), tuple(game.get_box_locations()), "")]
    mark = set()
    while stack:
        mike, boxes, path = stack.pop()
        if boxes == tuple(game.get_goal_locations()):
            return path
        s = (mike, boxes)
        if s in mark:
            continue
        mark.add(s)
        for d in dir_map:
            game.set_player_position(mike)
            game.set_box_positions(list(boxes))
            if game.apply_move(d):
                neew = (game.get_player_position(), tuple(game.get_box_locations()), path + d)
                stack.append(neew)
    return "No solution"


We implemented this algorithm using a stack so that when we see a node, we put all its neighbors in the stack and see them in depth and expand them.

In [229]:
path = dfs(game_map)
print(path)

RRRRLLLLDRRRRRRURRRRRRRRRRDLLLLLLLLLLLLLLLLDRRRRRRRRRRRRRRLLLLLLLLLLLLLLDRRRRDRDLLLLLLLLURRRRURRRRDLDRDLRDLLLLLLLLDRRRRRRRDRULLLLLLLLURRRRRRURRDLLLLLLLLDRRRRRRRDRULLLLLLLLURRRRURRRRDLLLLLLLLLLLLLLLLDRRRRRRRRRRRRRRLLLLLLLLLLLLLLDRRRRRDLLLLLLLLDRRRRRRRDRULLLLLLLLURRRRURRRRDLDRULLLLLLLLLLLLLLLLDRRRRRRRRRRRRRRDLLLDRRRRRRURRRRRRDRRRDRUULRDLLLLLLLLURRRRRRRRDLLLLLLLLLLLLLLLLDRRRRRRRRRRRRRRDRRDLLRRDLLLLLLLLURRRRRRLLLLLLDRRRRRRRRUULLLLLDRRRRRRURRRRRRRRRRDDLLLLLLLLLLLLLLLLDRRRRDRDLRDLLLLLLLLURRRRRRLLLLLLDRRRRRRRURUL


In [265]:
def ids(game_map):
    depth = 0
    while 1:
        mark = set()
        path = f_rec(Game(game_map), depth, mark, "")
        if path is not None:
            return path
        depth += 1

def f_rec(game, depth, mark, path):
    mike = game.get_player_position()
    boxes = tuple(game.get_box_locations())
    if boxes == tuple(game.get_goal_locations()):
        return path
    if depth == 0:
        return None
    s = (mike, boxes)
    if s in mark:
        return None
    mark.add(s)
    for d in dir_map:
        game.set_player_position(mike)
        game.set_box_positions(list(boxes))
        if game.apply_move(d):
            neew = f_rec(game, depth - 1, mark, path + d)
            if neew is not None:
                return neew
    mark.remove(s)
    return None

This algorithm is exactly like the previous algorithm, except that its depth is limited and we solve it recursively for all depths.

In [266]:
path = ids(game_map)
print(path)

UURDLDRRDRURDR


In [6]:
def MNH(first_point, second_point):
    x1 = first_point[0]
    x2 = first_point[1]
    y1 = second_point[0]
    y2 = second_point[1]
    manhatan = abs(x1 - y1) + abs(x2 - y2)
    return manhatan

def heuristic_1(game_map):
    game = Game(game_map)
    goals = game.get_goal_locations()
    def temp_function(now):
        ans = 0
        for i in range(len(now[1])):
            ans += MNH(now[0], now[1][i]) + MNH(now[1][i], goals[i])
        return ans
    return temp_function


def a_star_1(game_map):
    game = Game(game_map)
    temp_function = heuristic_1(game_map)
    H0 = (game.get_player_position(), tuple(game.get_box_locations()))
    h = temp_function(H0)
    goals = game.get_goal_locations()
    g = 0
    f = g + h
    nodes = [(f, g, H0, "")]
    mark = {H0: g}
    while nodes:
        node = heapq.heappop(nodes)
        f = node[0]
        g = node[1]
        s = node[2]
        path = node[3]
        mike, boxes = s
        if list(boxes) == goals:
            return path
        neighbors = []
        for d in dir_map:
            game_copy = Game(game_map)
            game_copy.set_player_position(mike)
            game_copy.set_box_positions(list(boxes))
            if game_copy.apply_move(d):
                neew = (game_copy.get_player_position(), tuple(game_copy.get_box_locations()))
                neighbors.append((d, neew))
        for d, neew in neighbors:
            now = g + 1
            if neew not in mark or now < mark[neew]:
                mark[neew] = now
                ff = now + temp_function(neew)
                heapq.heappush(nodes, (ff, now, neew, path + d))
                
    return None


In the function MNH we calculate the Euclidean distance between two points and in the heuristic function we use it and make the desired estimate for each situation.

In [413]:
path = a_star_1(game_map)
print(path)

URRRRRRRRRRRRRURDDR


In [3]:
def PO(p1, p2, portals):
    H0 = MNH(p1, p2)
    for portal in portals:
        H1 = min(MNH(p1, portal[0]) + MNH(portal[1], p2),MNH(p1, portal[1]) + MNH(portal[0], p2))
        H0 = min(H0, H1)
    return H0

def heuristic_2(game):
    goals = game.get_goal_locations()
    portals = game.get_portal_locations()
    def temp_function(s):
        mike, boxes = s
        if not boxes:
            return 0
        dis = []
        for box in boxes:
            distance = PO(mike, box, portals)
            dis.append(distance)
        mn1 = min(dis)
        mn2 = 0
        for i in range(len(boxes)):
            mn2 += PO(boxes[i], goals[i], portals)
        ans = mn1 + mn2
        return ans
    return temp_function


def a_star_2(game_map):
    game = Game(game_map)
    temp_function = heuristic_2(game)
    H0 = (game.get_player_position(), tuple(game.get_box_locations()))
    h = temp_function(H0)
    goals = game.get_goal_locations()
    g = 0
    f = g + h
    nodes = [(f, g, H0, "")]
    mark = {H0: g}
    while nodes:
        node = heapq.heappop(nodes)
        f = node[0]
        g = node[1]
        s = node[2]
        path = node[3]
        mike, boxes = s
        if list(boxes) == goals:
            return path
        neighbors = []
        for d in dir_map:
            game_copy = Game(game_map)
            game_copy.set_player_position(mike)
            game_copy.set_box_positions(list(boxes))
            if game_copy.apply_move(d):
                neew = (game_copy.get_player_position(), tuple(game_copy.get_box_locations()))
                neighbors.append((d, neew))
        for d, neew in neighbors:
            now = g + 1
            if neew not in mark or now < mark[neew]:
                mark[neew] = now
                ff = now + temp_function(neew)
                heapq.heappush(nodes, (ff, now, neew, path + d))
                
    return None



This algorithm is exactly the same as the previous one, except that in the Po function we check the Euclidean distance of two points based on whether they pass through the portals or not, and we assign a minimum state to each state using these distances.

In [370]:
path = a_star_2(game_map)
print(path)

UURDLDRRDRURDR


In [24]:
def weighted_a_star(game_map, w=4.15):
    game = Game(game_map)
    temp_function = heuristic_2(game)
    H0 = (game.get_player_position(), tuple(game.get_box_locations()))
    h = temp_function(H0)
    goals = game.get_goal_locations()
    g = 0
    f = g + w * h
    nodes = [(f, g, H0, "")]
    mark = {H0: g}
    while nodes:
        node = heapq.heappop(nodes)
        f = node[0]
        g = node[1]
        s = node[2]
        path = node[3]
        mike, boxes = s
        if list(boxes) == goals:
            return path
        neighbors = []
        for d in dir_map:
            game_copy = Game(game_map)
            game_copy.set_player_position(mike)
            game_copy.set_box_positions(list(boxes))
            if game_copy.apply_move(d):
                neew = (game_copy.get_player_position(), tuple(game_copy.get_box_locations()))
                neighbors.append((d, neew))
        for d, neew in neighbors:
            now = g + 1
            if neew not in mark or now < mark[neew]:
                mark[neew] = now
                ff = now + w * temp_function(neew)
                heapq.heappush(nodes, (ff, now, neew, path + d))
                
    return None

in this algorithm we have w as weiht for H(n)

In [26]:
path = weighted_a_star(game_map)
print(path)

RURURDRRDRDLDLULLULURRURDLUUUULLLURRURDDDDDLDRDLDRRDRURULULULDLDR


for w = 5 -> 11.7

for w = 4.5 -> 3.4

for w = 4.25 -> 2.8

for w = 4.15 -> 2.6

for w = 4 -> 3.4

for w = 3.5 -> 5.9

for w = 3 -> 16.6

for w = 2.5 -> 40.5

In [1]:
import random
import numpy as np
from math import inf
import pygame
import time
import math

def calculate_block_score(block):
    w = [0, 1, 10, 100, 1000]
    count_red = np.sum(block == 1)
    count_blue = np.sum(block == -1)
    if count_red > 0 and count_blue > 0:
        return 0
    if count_red > 0:
        player = -1 
        count = count_red
    elif count_blue > 0:
        player = 1  
        count = count_blue
    else:
        player = 0  
        count = 0
    score = player * w[count - 1]
    return score

class PentagoGame:
    def __init__(self, ui=False, print=False, depth=2):
        self.board = np.zeros((6, 6), dtype=int)
        self.current_player = 1
        self.ui = ui
        self.depth = depth
        self.nodes_visited = 0
        self.game_over = False
        self.result = None
        self.selected_block = None
        self.move_stage = 0  # 0: place piece, 1: select block, 2: rotate
        self.temp_piece = None
        self.print = print

        if ui:
            pygame.font.init()
            self.screen = pygame.display.set_mode((800, 600))
            pygame.display.set_caption("Pygame Board")
            # self.font = pygame.font.SysFont("Arial", 20)
            self.show_buttons = False
            self.buttons = {
                "rotate_cw": pygame.Rect(650, 200, 100, 50),
                "rotate_ccw": pygame.Rect(650, 300, 100, 50),
            }
            self.setup_controls()
            self.draw_board()

    def setup_controls(self):
        if self.show_buttons:
            pygame.draw.rect(self.screen, (144, 238, 144), self.buttons["rotate_cw"])   # Light Green
            pygame.draw.rect(self.screen, (173, 216, 230), self.buttons["rotate_ccw"])  # Light Blue

            self.screen.draw_text("CLOCKWISE", self.buttons["rotate_cw"].center)
            self.screen.draw_text("COUNTER-CLOCKWISE", self.buttons["rotate_ccw"].center)

    def hide_rotation_buttons(self):
        self.show_buttons = False

    def show_rotation_buttons(self):
        self.show_buttons = True

    def copy_board(self, board):
        return np.copy(board)

    def rotate_block(self, board, block, direction):
        row_start = (block // 2) * 3
        col_start = (block % 2) * 3
        sub = board[row_start : row_start + 3, col_start : col_start + 3]
        rotated = np.rot90(sub, 3 if direction == "cw" else 1)
        board[row_start : row_start + 3, col_start : col_start + 3] = rotated

    def get_possible_moves(self, board, player):
        moves = []
        for i in range(6):
            for j in range(6):
                if board[i][j] == 0:
                    for block in range(4):
                        for dir in ["cw", "ccw"]:
                            moves.append((i, j, block, dir))
        return moves

    def apply_move(self, board, move, player):
        new_board = self.copy_board(board)
        row, col, block, direction = move
        if new_board[row][col] != 0:
            return None
        new_board[row][col] = player
        self.rotate_block(new_board, block, direction)
        return new_board

    def check_winner(self, board):
        for i in range(6):
            for j in range(6):
                if board[i][j] == 0:
                    continue

                # Horizontal
                if j <= 1 and np.all(board[i, j : j + 5] == board[i][j]):
                    return board[i][j]

                # Vertical
                if i <= 1 and np.all(board[i : i + 5, j] == board[i][j]):
                    return board[i][j]

                # Diagonal
                if (
                    i <= 1
                    and j <= 1
                    and all(board[i + k][j + k] == board[i][j] for k in range(5))
                ):
                    return board[i][j]

                # Anti-diagonal
                if (
                    i <= 1
                    and j >= 4
                    and all(board[i + k][j - k] == board[i][j] for k in range(5))
                ):
                    return board[i][j]
        if np.all(board != 0):
            return 0
        return None

    def main_hurestic(self, board):
        res = 0
        for i in range(6):
            for j in range(2):
                block = board[i, j:j+5]
                res += calculate_block_score(block)

        for j in range(6):
            for i in range(2):
                block = board[i:i+5, j]
                res += calculate_block_score(block)

        for i in range(2):
            for j in range(2):
                block = []
                for k in range(5):
                    row = i + k
                    col = j + k
                    block.append(board[row][col])
                res += calculate_block_score(block)

        for i in range(2):
            for j in range(4, 6):
                block = []
                for k in range(5):
                    row = i + k
                    col = j - k
                    block.append(board[row][col])
                res += calculate_block_score(block)

        return res

    def hurestic_with_rotation(self, board):
        best = -inf
        for block in range(4):
            for direction in ('cw', 'ccw'):
                b2 = self.copy_board(board)
                self.rotate_block(b2, block, direction)
                val = self.main_hurestic(b2)
                best = max(best, val)
        return best

    def minimax(self, board, depth, prun, maximizing, a=-inf, b=inf):
        self.nodes_visited += 1
        winner = self.check_winner(board)
        if depth == 0 or winner is not None:
            # base = self.main_hurestic(board)
            rot  = self.hurestic_with_rotation(board)
            # return base, None
            return rot, None

        if maximizing:
            value, best_move = -inf, None
            for move in self.get_possible_moves(board, -1):
                new_board = self.apply_move(board, move, -1)
                if new_board is None:
                    continue
                if prun:
                    val, unuseablevalue = self.minimax(new_board, depth-1, prun, False, a, b)
                    a = max(a, val)
                    if b <= a:
                        break
                else:
                    val, unuseablevalue = self.minimax(new_board, depth-1, prun, False)
                if val > value:
                    value, best_move = val, move
            return value, best_move
        else:
            value, best_move = inf, None
            for move in self.get_possible_moves(board, 1):
                new_board = self.apply_move(board, move, 1)
                if new_board is None:
                    continue
                if prun:
                    val, unuseablevalue = self.minimax(new_board, depth-1, prun, True, a, b)
                    b = min(b, val)
                    if b <= a:
                        break
                else:
                    val, unuseablevalue = self.minimax(new_board, depth-1, prun, True)
                if val < value:
                    value, best_move = val, move
            return value, best_move
        
            #TODO: Implement minmax algorithm

    def get_computer_move(self):
        start = time.time()
        self.nodes_visited = 0
        best_value = -inf
        best_move = None

        moves = self.get_possible_moves(self.board, -1)
        for move in moves:
            if self.game_over:
                break
            new_board = self.apply_move(self.board, move, -1)
            if new_board is None:
                continue

            try:
                value, unuseablevalue = self.minimax(new_board,self.depth - 1,prun=False,maximizing=False,a=best_value,b=inf)
                #TODO: Implement alpha-beta pruning algorithm

            except:
                value = -inf

            if value > best_value:
                best_value = value
                best_move = move

        if self.print:
            print(f"Move took {time.time() - start:.2f}s, nodes visited: {self.nodes_visited}")
        self.nodes_visited = 0
        return best_move

    def draw_text(self, text, center_pos, max_width):
        font_size = 24
        font = pygame.font.Font(None, font_size)
        text_surface = font.render(text, True, (0, 0, 0))

        text_width = text_surface.get_width()
        if text_width > max_width:
            scale_factor = max_width / text_width
            new_font_size = int(font_size * scale_factor)
            font = pygame.font.Font(None, new_font_size)
            text_surface = font.render(text, True, (0, 0, 0))

        text_rect = text_surface.get_rect(center=center_pos)
        self.screen.blit(text_surface, text_rect)

    def draw_board(self):
        self.screen.fill((0, 0, 0))

        for i in range(6):
            for j in range(6):
                x0 = j * 100
                y0 = i * 100

                if self.board[i][j] == 1:
                    pygame.draw.circle(self.screen, (255, 0, 0), (x0 + 50, y0 + 50), 40)
                elif self.board[i][j] == -1:
                    pygame.draw.circle(self.screen, (0, 0, 255), (x0 + 50, y0 + 50), 40)

                pygame.draw.rect(self.screen, (255, 255, 255), (x0, y0, 100, 100), 1)

        for i in [3, 6]:
            pygame.draw.line(self.screen, (255, 255, 255), (0, i * 100), (600, i * 100), 3)  # Horizontal
            pygame.draw.line(self.screen, (255, 255, 255), (i * 100, 0), (i * 100, 600), 3)  # Vertical

        # Show rotation buttons if in move_stage 2
        if self.move_stage == 2:
            self.highlight_selected_block()
            self.show_rotation_buttons()

        if self.show_buttons:
            pygame.draw.rect(self.screen, (144, 238, 144), self.buttons["rotate_cw"])  # Light Green
            pygame.draw.rect(self.screen, (173, 216, 230), self.buttons["rotate_ccw"])  # Light Blue

            self.draw_text(
                "CLOCKWISE",
                self.buttons["rotate_cw"].center,
                self.buttons["rotate_cw"].width,
            )
            self.draw_text(
                "COUNTER-CLOCKWISE",
                self.buttons["rotate_ccw"].center,
                self.buttons["rotate_ccw"].width,
            )

    def click_handler(self, event):
        if self.game_over or self.current_player != 1:
            return

        x, y = event.pos
        if self.move_stage == 0:  # Place piece
            if x > 600:
                return  # clicks on control area
            col = x // 100
            row = y // 100
            if 0 <= row < 6 and 0 <= col < 6 and self.board[row][col] == 0:
                self.temp_piece = (row, col)
                self.board[row][col] = 1
                self.move_stage = 1
                self.draw_board()

        elif self.move_stage == 1:  # Select block
            if x > 600:
                return
            # which block was clicked
            block_x = 0 if x < 300 else 1
            block_y = 0 if y < 300 else 1
            self.selected_block = block_y * 2 + block_x
            self.move_stage = 2
            self.show_rotation_buttons()
            self.highlight_selected_block()

        elif self.move_stage == 2:  # Rotate
            if self.buttons["rotate_cw"].collidepoint(event.pos):
                self.apply_rotation("cw")
            if self.buttons["rotate_ccw"].collidepoint(event.pos):
                self.apply_rotation("ccw")

    def apply_rotation(self, direction):
        self.rotate_block(self.board, self.selected_block, direction)
        self.current_player = -1
        self.move_stage = 0
        self.selected_block = None
        self.temp_piece = None
        self.hide_rotation_buttons()
        self.draw_board()
        pygame.display.flip()
        self.check_game_over()
        pygame.time.delay(1000)
        self.play_computer_move()

    def highlight_selected_block(self):
        colors = [
            (255, 153, 153),
            (153, 255, 153),
            (153, 153, 255),
            (255, 255, 153),
        ]  # RGB colors

        row_start = (self.selected_block // 2) * 3
        col_start = (self.selected_block % 2) * 3

        pygame.draw.rect(
            self.screen,
            colors[self.selected_block],
            (col_start * 100, row_start * 100, 300, 300),
            5,
        )

    def play_computer_move(self):
        move = self.get_computer_move()
        if move and not self.game_over:
            new_board = self.apply_move(self.board, move, -1)
            if new_board is not None:
                self.board = new_board
                self.current_player = 1
                self.draw_board()
                pygame.display.flip()
                self.check_game_over()
            else:
                print("Invalid computer move!")

    def check_game_over(self):
        winner = self.check_winner(self.board)
        if winner is not None:
            self.game_over = True
            self.result = winner
            print("Game over! Result:", winner)
            if self.ui:
                self.show_game_over_message()

    def show_game_over_message(self):
        self.screen.fill((200, 200, 200))
        pygame.draw.rect(self.screen, (255, 255, 255), (100, 200, 500, 200))
        pygame.draw.rect(self.screen, (0, 0, 0), (100, 200, 500, 200), 3)

        result_text = f"Player {self.result} wins!" if self.result != 0 else "Draw!"
        text_surface = self.font_large.render(result_text, True, (255, 0, 0))
        self.screen.blit(text_surface, (250, 250))

        exit_text = self.font_small.render("Click anywhere to exit", True, (0, 0, 0))
        self.screen.blit(exit_text, (230, 350))
        pygame.display.flip()

    def play(self):
        if self.ui:
            running = True
            while running:
                for event in pygame.event.get():
                    if event.type == pygame.QUIT:
                        running = False
                    elif event.type == pygame.MOUSEBUTTONDOWN:
                        self.click_handler(event)
                self.draw_board()
                pygame.display.flip()
            pygame.quit()
            return self.result
        else:
            while not self.game_over:
                self.print_board()
                winner = self.check_winner(self.board)
                if winner is not None:
                    return winner

                if self.current_player == 1:
                    move = random.choice(self.get_possible_moves(self.board, 1))
                else:
                    move = self.get_computer_move()

                self.board = self.apply_move(self.board, move, self.current_player)
                self.current_player *= -1
            return self.result

    def print_board(self):
        if self.print == False:
            return
        print("-" * 25)
        for row in self.board:
            print(" ".join(f"{x:2}" for x in row))
        print("-" * 25)
        

pygame 2.6.1 (SDL 2.28.4, Python 3.11.5)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [ ]:
if __name__ == "__main__":
    numGames = 1
    numWins, numTies, numLosses = 0, 0, 0
    for i in range(numGames):
        game = PentagoGame(ui=False, print=True, depth=2)  # depth=2 for faster
        result = game.play()
        if result == -1:
            numWins += 1
        elif result == 0:
            numTies += 1
        else:
            numLosses += 1

    print(f"{numWins} wins, {numTies} ties, {numLosses} losses")

-------------------------
 0  0  0  0  0  0
 0  0  0  0  0  0
 0  0  0  0  0  0
 0  0  0  0  0  0
 0  0  0  0  0  0
 0  0  0  0  0  0
-------------------------
-------------------------
 0  0  0  0  0  0
 0  0  0  0  0  0
 0  0  0  0  0  0
 0  0  0  1  0  0
 0  0  0  0  0  0
 0  0  0  0  0  0
-------------------------
Move took 164.12s, nodes visited: 76440
-------------------------
 0  0 -1  0  0  0
 0  0  0  0  0  0
 0  0  0  0  0  0
 0  0  0  1  0  0
 0  0  0  0  0  0
 0  0  0  0  0  0
-------------------------
-------------------------
 0  0 -1  0  0  0
 0  0  0  0  0  0
 0  0  0  0  0  0
 0  0  0  1  0  0
 0  0  0  0  0  0
 0  0  0  0  0  1
-------------------------
